# 📖 Notebook 3: Search Ranking and Relevance

Finding posts that *match* a query is only half the problem. The other half is **ranking** — showing the *best* results first. When you search "Taylor Swift" on Facebook, you don't want a random post from 5 years ago with 0 likes. You want the most relevant, popular, and recent results.

In this notebook, we'll explore how search engines score and rank results, and how to combine relevance with business signals like recency and popularity.

## Learning Objectives

By the end of this notebook, you'll understand:
- How **TF-IDF** and **BM25** score relevance
- How to sort by **recency** or **like count** in Elasticsearch
- How **function_score** queries combine relevance with business signals
- How **multi-keyword** and **phrase queries** work
- The **two-stage architecture** pattern used in production search systems

## 🛠️ Setup

Make sure infrastructure is running:

```bash
cd 06-system-designs/fb-post-search
docker compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
from elasticsearch import Elasticsearch, helpers
import time
import json
import pandas as pd

DB_CONFIG = {
    "host": "localhost",
    "port": 55433,
    "database": "fb_post_search",
    "user": "demo",
    "password": "demo"
}

es = Elasticsearch("http://localhost:9200")

def get_db():
    return psycopg2.connect(**DB_CONFIG)

# Verify connections
try:
    conn = get_db()
    conn.close()
    print("✅ PostgreSQL connected")
except Exception as e:
    print(f"❌ PostgreSQL: {e}")

try:
    es.info()
    print("✅ Elasticsearch connected")
except Exception as e:
    print(f"❌ Elasticsearch: {e}")

In [ ]:
# Ensure the 'posts' index exists (created in Notebook 1)
# If not, re-create and populate it

INDEX_NAME = "posts"

if not es.indices.exists(index=INDEX_NAME):
    print("Index 'posts' not found — creating it now...")
    es.indices.create(
        index=INDEX_NAME,
        body={
            "settings": {
                "number_of_shards": 1,
                "number_of_replicas": 0,
                "analysis": {
                    "analyzer": {
                        "post_analyzer": {
                            "type": "custom",
                            "tokenizer": "standard",
                            "filter": ["lowercase", "stop", "snowball"]
                        }
                    }
                }
            },
            "mappings": {
                "properties": {
                    "content":    {"type": "text", "analyzer": "post_analyzer"},
                    "user_id":    {"type": "integer"},
                    "like_count": {"type": "integer"},
                    "created_at": {"type": "date"}
                }
            }
        }
    )

    conn = get_db()
    cur = conn.cursor()
    cur.execute("SELECT id, user_id, content, like_count, created_at FROM posts")
    rows = cur.fetchall()
    conn.close()

    actions = [{
        "_index": INDEX_NAME, "_id": r[0],
        "_source": {"user_id": r[1], "content": r[2], "like_count": r[3],
                    "created_at": r[4].isoformat() if r[4] else None}
    } for r in rows]
    helpers.bulk(es, actions)
    es.indices.refresh(index=INDEX_NAME)
    print(f"✅ Indexed {len(actions)} posts")
else:
    count = es.count(index=INDEX_NAME)["count"]
    print(f"✅ Index 'posts' exists with {count} documents")

## 📐 How Search Scoring Works: TF-IDF and BM25

When you search for "coffee", Elasticsearch gives each matching post a **score**. Higher scores = shown first. But how does it decide which post is more relevant?

### TF-IDF (the foundation)

**TF-IDF** stands for **Term Frequency × Inverse Document Frequency**:

- **Term Frequency (TF)**: How often does "coffee" appear in *this* post?  
  → A post mentioning "coffee" 5 times is probably more about coffee than one mentioning it once.

- **Inverse Document Frequency (IDF)**: How rare is "coffee" across *all* posts?  
  → If "coffee" appears in 5 out of 500 posts, it's a meaningful search term.  
  → If "the" appears in 495 out of 500 posts, it's useless for ranking.

```
TF-IDF score = TF × IDF
             = (times word appears in doc) × log(total docs / docs with word)
```

### BM25 (the upgrade)

**BM25** (Best Matching 25) is TF-IDF with two improvements:

1. **Saturation**: After a word appears many times, the score increase slows down. Mentioning "coffee" 50 times isn't 50× more relevant than mentioning it once.

2. **Document length normalization**: A short post that mentions "coffee" once is probably more focused on coffee than a 1000-word essay that mentions it once.

**Elasticsearch uses BM25 by default.** You rarely need to change this.

In [ ]:
# Let's see BM25 in action — search for "coffee" and examine scores

result = es.search(
    index=INDEX_NAME,
    body={
        "query": {"match": {"content": "coffee"}},
        "explain": True,
        "size": 5
    }
)

print("☕ BM25 Scores for 'coffee' search")
print("=" * 70)

for hit in result["hits"]["hits"]:
    score = hit["_score"]
    content = hit["_source"]["content"][:70]
    likes = hit["_source"]["like_count"]

    # Extract BM25 components from explanation
    explanation = hit["_explanation"]
    print(f"\n  Score: {score:.4f} | {likes} likes")
    print(f"  Content: {content}...")
    print(f"  Scoring: {explanation['description'][:80]}")

print()
print("💡 Posts that mention 'coffee' more often and are shorter get higher BM25 scores.")
print("   But notice: BM25 ignores likes and recency entirely!")

## 📅 Sorting by Recency

The Facebook Post Search design requires sorting by **recency** (newest first) or **like count** (most popular first). BM25 only measures text relevance — we need to add our own sorting.

The simplest approach: ignore BM25 scoring and sort by a field.

In [ ]:
# Sort by recency (newest posts first)

def search_by_recency(keyword, size=10):
    """Find matching posts, sorted by newest first."""
    return es.search(
        index=INDEX_NAME,
        body={
            "query": {"match": {"content": keyword}},
            "sort": [{"created_at": {"order": "desc"}}],
            "size": size
        }
    )

result = search_by_recency("coffee")

print("📅 Search 'coffee' — sorted by RECENCY (newest first)")
print("=" * 70)
for hit in result["hits"]["hits"]:
    src = hit["_source"]
    date = src["created_at"][:10] if src["created_at"] else "unknown"
    print(f"  [{date}] {src['like_count']:>5} likes | {src['content'][:55]}...")

In [ ]:
# Sort by like count (most popular first)

def search_by_likes(keyword, size=10):
    """Find matching posts, sorted by most liked."""
    return es.search(
        index=INDEX_NAME,
        body={
            "query": {"match": {"content": keyword}},
            "sort": [{"like_count": {"order": "desc"}}],
            "size": size
        }
    )

result = search_by_likes("coffee")

print("👍 Search 'coffee' — sorted by LIKES (most popular first)")
print("=" * 70)
for hit in result["hits"]["hits"]:
    src = hit["_source"]
    date = src["created_at"][:10] if src["created_at"] else "unknown"
    print(f"  [{date}] {src['like_count']:>5} likes | {src['content'][:55]}...")

## 🎯 Combining Relevance + Popularity + Recency

In the real world, you don't want *just* relevance, *just* recency, or *just* likes. You want a **blend**:

- A post from 10 minutes ago with 5 likes about "coffee" should beat
- A post from 2 years ago with 100 likes that barely mentions "coffee"

Elasticsearch's **`function_score`** query lets you combine BM25 with custom scoring functions. Think of it as:

```
final_score = BM25_score × f(like_count) × f(recency)
```

This is the heart of how Facebook and other platforms rank search results.

In [ ]:
# function_score: Boost by like count
# Uses log1p(like_count) so popular posts score higher,
# but the boost saturates (1000 likes isn't 1000× better than 1 like)

def search_boosted_by_likes(keyword, size=10):
    """BM25 relevance boosted by popularity (like count)."""
    return es.search(
        index=INDEX_NAME,
        body={
            "query": {
                "function_score": {
                    "query": {"match": {"content": keyword}},
                    "functions": [
                        {
                            "field_value_factor": {
                                "field": "like_count",
                                "modifier": "log1p",    # log(1 + like_count)
                                "factor": 2             # weight of this signal
                            }
                        }
                    ],
                    "boost_mode": "multiply"   # final = BM25 × like_boost
                }
            },
            "size": size
        }
    )

result = search_boosted_by_likes("coffee")

print("🎯 Search 'coffee' — BM25 × Popularity Boost")
print("=" * 70)
for hit in result["hits"]["hits"]:
    src = hit["_source"]
    date = src["created_at"][:10] if src["created_at"] else "unknown"
    print(f"  score={hit['_score']:>8.2f} | [{date}] {src['like_count']:>5} likes | {src['content'][:45]}...")

print()
print("💡 Now popular posts rank higher, but relevance still matters.")
print("   A very relevant post with few likes can still beat a mildly")
print("   relevant post with many likes.")

In [ ]:
# function_score: Boost by recency using exponential decay
# Posts lose score as they age. Recent posts get a big boost.

def search_boosted_by_recency(keyword, size=10):
    """BM25 relevance boosted by recency (newer = higher score)."""
    return es.search(
        index=INDEX_NAME,
        body={
            "query": {
                "function_score": {
                    "query": {"match": {"content": keyword}},
                    "functions": [
                        {
                            "exp": {
                                "created_at": {
                                    "origin": "now",     # center = right now
                                    "scale": "7d",       # half-life = 7 days
                                    "decay": 0.5         # score halves every 7 days
                                }
                            }
                        }
                    ],
                    "boost_mode": "multiply"
                }
            },
            "size": size
        }
    )

result = search_boosted_by_recency("coffee")

print("📅 Search 'coffee' — BM25 × Recency Decay")
print("=" * 70)
for hit in result["hits"]["hits"]:
    src = hit["_source"]
    date = src["created_at"][:10] if src["created_at"] else "unknown"
    print(f"  score={hit['_score']:>8.4f} | [{date}] {src['like_count']:>5} likes | {src['content'][:45]}...")

print()
print("💡 Exponential decay means:")
print("   - Posts from today: ~100% of BM25 score")
print("   - Posts from 7 days ago: ~50% of BM25 score")
print("   - Posts from 14 days ago: ~25% of BM25 score")
print("   - Posts from 30 days ago: ~6% of BM25 score")

In [ ]:
# The full combo: BM25 × popularity × recency

def search_combined_ranking(keyword, size=10):
    """BM25 relevance boosted by both popularity AND recency."""
    return es.search(
        index=INDEX_NAME,
        body={
            "query": {
                "function_score": {
                    "query": {"match": {"content": keyword}},
                    "functions": [
                        {
                            "field_value_factor": {
                                "field": "like_count",
                                "modifier": "log1p",
                                "factor": 1.5
                            }
                        },
                        {
                            "exp": {
                                "created_at": {
                                    "origin": "now",
                                    "scale": "14d",
                                    "decay": 0.5
                                }
                            }
                        }
                    ],
                    "score_mode": "multiply",   # combine functions by multiplying
                    "boost_mode": "multiply"     # combine with BM25 by multiplying
                }
            },
            "size": size
        }
    )

result = search_combined_ranking("coffee")

print("🏆 Search 'coffee' — BM25 × Popularity × Recency")
print("=" * 70)
for hit in result["hits"]["hits"]:
    src = hit["_source"]
    date = src["created_at"][:10] if src["created_at"] else "unknown"
    print(f"  score={hit['_score']:>8.2f} | [{date}] {src['like_count']:>5} likes | {src['content'][:45]}...")

print()
print("💡 This is the 'production recipe': BM25 × popularity × recency.")
print("   Tune the weights (factor, scale, decay) based on your product needs.")

## 🔗 Multi-Keyword and Phrase Queries

Users don't always search for single words. What happens when someone searches for "Taylor Swift"?

There are three ways to handle this:

1. **OR match**: Posts containing "Taylor" OR "Swift" (most results, lowest precision)
2. **AND match**: Posts containing "Taylor" AND "Swift" (fewer results, higher precision)
3. **Phrase match**: Posts containing the exact phrase "Taylor Swift" (fewest results, highest precision)

In [ ]:
# Compare: OR vs AND vs Phrase matching for "Taylor Swift"

# OR match (default) — either word matches
or_result = es.search(
    index=INDEX_NAME,
    body={"query": {"match": {"content": "Taylor Swift"}}, "size": 10}
)

# AND match — both words must appear (but not necessarily together)
and_result = es.search(
    index=INDEX_NAME,
    body={"query": {"match": {"content": {"query": "Taylor Swift", "operator": "and"}}}, "size": 10}
)

# Phrase match — words must appear together in order
phrase_result = es.search(
    index=INDEX_NAME,
    body={"query": {"match_phrase": {"content": "Taylor Swift"}}, "size": 10}
)

print("🔗 Multi-Keyword Query: 'Taylor Swift'")
print("=" * 70)
print(f"  OR match:     {or_result['hits']['total']['value']} results (Taylor OR Swift)")
print(f"  AND match:    {and_result['hits']['total']['value']} results (Taylor AND Swift)")
print(f"  Phrase match: {phrase_result['hits']['total']['value']} results (exact phrase)")

print("\n--- OR match results (notice some are about 'swift' but not Taylor Swift) ---")
for hit in or_result["hits"]["hits"][:3]:
    print(f"  [score={hit['_score']:.2f}] {hit['_source']['content'][:65]}...")

print("\n--- Phrase match results (exact 'Taylor Swift') ---")
for hit in phrase_result["hits"]["hits"][:3]:
    print(f"  [score={hit['_score']:.2f}] {hit['_source']['content'][:65]}...")

In [ ]:
# Best practice: use bool query to combine phrase + OR for best results
# Phrase matches get a big boost, but OR matches still appear

def search_smart_multi_keyword(query_text, size=10):
    """Smart multi-keyword search: boost exact phrases, include partial matches."""
    return es.search(
        index=INDEX_NAME,
        body={
            "query": {
                "bool": {
                    "must": [
                        {"match": {"content": query_text}}
                    ],
                    "should": [
                        {"match_phrase": {"content": {"query": query_text, "boost": 3}}}
                    ]
                }
            },
            "size": size
        }
    )

result = search_smart_multi_keyword("Taylor Swift")

print("🧠 Smart Search: 'Taylor Swift' (phrase boosted 3×)")
print("=" * 70)
for hit in result["hits"]["hits"]:
    src = hit["_source"]
    print(f"  score={hit['_score']:>6.2f} | {src['like_count']:>5} likes | {src['content'][:55]}...")

print()
print("💡 Posts with the exact phrase 'Taylor Swift' rank highest,")
print("   but posts mentioning either word still appear.")

## 🏗️ Two-Stage Architecture

In the FB Post Search design, the like counts in the index may be **stale** (to reduce write volume, likes are only updated at milestones like powers of 2). This leads to a **two-stage architecture**:

```
Stage 1: RETRIEVE (fast, approximate)           Stage 2: RE-RANK (precise)
┌─────────────────────────────────┐     ┌────────────────────────────────────┐
│  Elasticsearch                  │     │  Search Service                    │
│                                 │     │                                    │
│  Query: "coffee"                │     │  For each of 20 candidates:        │
│  Get top 2N results using       │────→│  1. Fetch real-time like count     │
│  approximate like counts        │     │  2. Re-sort by actual like count   │
│                                 │     │  3. Return top N to user           │
└─────────────────────────────────┘     └────────────────────────────────────┘
```

**Why 2N?** We fetch more than we need because the approximate ranking might be slightly wrong. After re-ranking with fresh data, we take the top N.

This pattern is extremely common in information retrieval and recommendation systems. Let's implement it.

In [ ]:
# ── Two-stage retrieve-and-rerank ──────────────────────────────────────────
#
# The premise is that the like_count stored in Elasticsearch is STALE. In the
# design, likes are only pushed to the index at milestones (1, 2, 4, 8, 16 …)
# so that 100K likes/sec doesn't become 100K index writes/sec.
#
# Right now our ES like_counts came straight from PostgreSQL, so they agree
# perfectly and a re-rank would be a no-op — it would look like stage 2 earns
# its keep when it demonstrably doesn't. So first, let's create real staleness:
# some posts go viral in PostgreSQL, and the index doesn't hear about it.

KEYWORD = "coffee"


def es_candidates(keyword, size):
    """Stage 1: cheap retrieval, ranked by the index's (possibly stale) likes."""
    res = es.search(
        index=INDEX_NAME,
        body={
            "query": {"match": {"content": keyword}},
            "sort": [{"like_count": {"order": "desc"}}],
            "size": size,
        },
    )
    return [
        {"post_id": int(h["_id"]),
         "content": h["_source"]["content"],
         "es_likes": h["_source"]["like_count"]}
        for h in res["hits"]["hits"]
    ]


# ── Create the staleness: promote two mid-ranked posts in PostgreSQL only ──
pool = es_candidates(KEYWORD, 10)
top_likes = pool[0]["es_likes"]
went_viral = pool[6:8]          # currently ranked 7th and 8th by the index

conn = get_db()
conn.autocommit = True
cur = conn.cursor()
original_likes = {}
for offset, cand in enumerate(went_viral):
    cur.execute("SELECT like_count FROM posts WHERE id = %s", (cand["post_id"],))
    original_likes[cand["post_id"]] = cur.fetchone()[0]
    # These two just got hugged by the front page. PostgreSQL knows. ES doesn't.
    cur.execute("UPDATE posts SET like_count = %s WHERE id = %s",
                (top_likes + 5000 + offset, cand["post_id"]))
conn.close()

print("📈 Simulated: posts "
      f"{[c['post_id'] for c in went_viral]} went viral since the last index update.\n")


def two_stage_search(keyword, n=5):
    """Stage 1: 2N candidates from ES.  Stage 2: re-rank on live like counts."""
    candidates = es_candidates(keyword, n * 2)

    conn = get_db()
    cur = conn.cursor()
    cur.execute("SELECT id, like_count FROM posts WHERE id = ANY(%s)",
                ([c["post_id"] for c in candidates],))
    live = dict(cur.fetchall())
    conn.close()

    for c in candidates:
        c["real_likes"] = live.get(c["post_id"], c["es_likes"])

    candidates.sort(key=lambda x: -x["real_likes"])
    return candidates[:n]


# ── Compare stage-1-only against the full two-stage result ─────────────────
N = 5
stage1_only = es_candidates(KEYWORD, N)
start = time.time()
final = two_stage_search(KEYWORD, n=N)
elapsed = (time.time() - start) * 1000

print(f"🏗️ Two-Stage Search for '{KEYWORD}'  ({elapsed:.1f}ms total)")
print("=" * 78)
print(f"  {'#':<3} {'Stage 1 only (stale index)':<36} {'After re-rank (live likes)'}")
print("-" * 78)
for i in range(N):
    a, b = stage1_only[i], final[i]
    print(f"  {i+1:<3} post {a['post_id']:<5} {a['es_likes']:>7} likes"
          f"{'':<12} post {b['post_id']:<5} {b['real_likes']:>7} likes")

moved = [c["post_id"] for c in final if c["post_id"] not in
         [s["post_id"] for s in stage1_only]]
print(f"\n  Posts the re-rank pulled into the top {N}: {moved}")
assert moved, "re-rank changed nothing — the staleness setup did not take effect"

print(f"""
  Stage 1 ranked those two posts 7th and 8th because that is what the index
  believed. Stage 2 asked PostgreSQL for the truth and moved them to the top.

  ⚖️  The cost: stage 2 is a second datastore hit on every search, and it can
     only reorder what stage 1 returned. Fetch 2N and a post ranked 3N by the
     stale index is invisible no matter how viral it went. Widening the
     candidate window fixes that and costs latency — that trade-off (how deep
     to retrieve before re-ranking) is the whole tuning exercise in production.

  ⚖️  Also note stage 2 re-ranks on likes ALONE here, throwing away BM25
     relevance. A real re-ranker blends the fresh signal back into the original
     score rather than replacing it.""")

# ── Restore the like counts so the rest of the notebook is unaffected ──────
conn = get_db()
conn.autocommit = True
cur = conn.cursor()
for pid, likes in original_likes.items():
    cur.execute("UPDATE posts SET like_count = %s WHERE id = %s", (likes, pid))
conn.close()
print("\n🔄 Restored original like counts.")

## 📄 Paginating a Result Set That Won't Hold Still

Everything so far returned the top N. Real search has a "next page" button, and that is
where a write-heavy system gets interesting.

Facebook takes **10,000 new posts per second**. Between the moment a user sees page 1 and
the moment they click "next", thousands of new posts have been indexed. The naive
`from` / `size` pagination asks the index a fresh question each time:

```
page 1  →  from=0, size=5   →  "give me results 1-5 of whatever matches right now"
   ...user reads for 3 seconds, 30,000 new posts arrive...
page 2  →  from=5, size=5   →  "give me results 6-10 of whatever matches NOW"
```

Those are two different result sets. If the new posts sort *above* the old ones — which they
do, if you sort by recency — everything shifts down and page 2 re-shows rows the user
already read.

Two separate problems hide here, and they have different fixes:

| Problem | Symptom | Fix |
|---------|---------|-----|
| **Result-set drift** | duplicate (or skipped) rows across pages | `search_after` — a cursor, not an offset |
| **Deep paging cost** | `from=100000` gets slow, then refuses | there is no fix; don't offer page 10,000 |

Let's reproduce the first one.

In [ ]:
# ── A dedicated index so this section can't disturb the others ─────────────
from datetime import datetime, timedelta

PAGING_INDEX = "posts_paging"
PAGE_SIZE = 5

if es.indices.exists(index=PAGING_INDEX):
    es.indices.delete(index=PAGING_INDEX)

es.indices.create(
    index=PAGING_INDEX,
    body={
        "settings": {"number_of_shards": 1, "number_of_replicas": 0},
        "mappings": {
            "properties": {
                "post_id": {"type": "integer"},
                "content": {"type": "text"},
                # A date we control, so "newest first" is deterministic.
                "created_at": {"type": "date"},
            }
        },
    },
)

T0 = datetime(2026, 1, 1, 12, 0, 0)

# 30 existing posts, post_id 1 newest .. 30 oldest.
helpers.bulk(es, [
    {"_index": PAGING_INDEX, "_id": i,
     "_source": {"post_id": i, "content": f"coffee post number {i}",
                 "created_at": (T0 - timedelta(minutes=i)).isoformat()}}
    for i in range(1, 31)
])
es.indices.refresh(index=PAGING_INDEX)

# Sorting must be a TOTAL order. created_at alone can tie, and ties break
# arbitrarily between shards and refreshes — which is its own source of
# duplicate rows. post_id is the tiebreaker.
SORT = [{"created_at": "desc"}, {"post_id": "desc"}]
QUERY = {"match": {"content": "coffee"}}

print(f"✅ {PAGING_INDEX}: 30 posts, newest = post 1")

In [ ]:
# ── The broken way: from / size ────────────────────────────────────────────

def page_offset(page_number, size=PAGE_SIZE):
    res = es.search(index=PAGING_INDEX, query=QUERY, sort=SORT,
                    from_=(page_number - 1) * size, size=size)
    return [h["_source"]["post_id"] for h in res["hits"]["hits"]]


page1 = page_offset(1)
print(f"  User loads page 1  →  posts {page1}")

# ── 3 new posts arrive while the user is reading ───────────────────────────
helpers.bulk(es, [
    {"_index": PAGING_INDEX, "_id": 100 + j,
     "_source": {"post_id": 100 + j, "content": "coffee breaking news",
                 "created_at": (T0 + timedelta(minutes=j)).isoformat()}}
    for j in range(1, 4)
])
es.indices.refresh(index=PAGING_INDEX)
print("  … 3 newer posts are indexed while the user reads …")

page2 = page_offset(2)
print(f"  User clicks next   →  posts {page2}")

dupes = sorted(set(page1) & set(page2))
print(f"\n  ❌ Posts shown twice: {dupes}")
assert dupes, "expected the offset window to shift and duplicate rows"
print(f"     {len(dupes)} of {PAGE_SIZE} rows on page 2 were already on page 1.")
print("     3 new posts pushed everything down by exactly 3 positions, and")
print("     `from=5` still means 'skip 5', just of a different list.")
print("\n     Worse in the other direction: if posts are *deleted* between pages,")
print("     the window shifts up and the user silently never sees those rows.")

### The fix: `search_after` — remember *where you were*, not *how far in*

`from=5` says "skip the first five of whatever matches now". That is a position in a list
that keeps changing.

`search_after` says "give me what comes after **this exact sort key**". The sort key is a
property of the last row the user actually saw, so new arrivals above it simply don't
matter — they land on page 1 territory, not in the middle of the user's reading.

```python
res  = es.search(..., sort=SORT, size=5)
last = res["hits"]["hits"][-1]["sort"]      # e.g. [1767268500000, 5]
next = es.search(..., sort=SORT, size=5, search_after=last)
```

This is the same idea as keyset pagination in SQL
(`WHERE (created_at, id) < (:last_created_at, :last_id) ORDER BY ... LIMIT 5`), and it needs
the same thing to be correct: the sort must be a **total order**. A tiebreaker column is not
optional — with ties, "after this key" is ambiguous and you're back to skipping rows.

In [ ]:
# ── The right way: search_after ────────────────────────────────────────────

# Rebuild the original 30 posts so both approaches start from the same state.
es.indices.delete(index=PAGING_INDEX)
es.indices.create(index=PAGING_INDEX, body={
    "settings": {"number_of_shards": 1, "number_of_replicas": 0},
    "mappings": {"properties": {"post_id": {"type": "integer"},
                                "content": {"type": "text"},
                                "created_at": {"type": "date"}}}})
helpers.bulk(es, [
    {"_index": PAGING_INDEX, "_id": i,
     "_source": {"post_id": i, "content": f"coffee post number {i}",
                 "created_at": (T0 - timedelta(minutes=i)).isoformat()}}
    for i in range(1, 31)])
es.indices.refresh(index=PAGING_INDEX)


def page_cursor(after=None, size=PAGE_SIZE):
    """Return (post_ids, cursor_for_next_page)."""
    kwargs = {"index": PAGING_INDEX, "query": QUERY, "sort": SORT, "size": size}
    if after is not None:
        kwargs["search_after"] = after
    res = es.search(**kwargs)
    hits = res["hits"]["hits"]
    if not hits:
        return [], None
    return [h["_source"]["post_id"] for h in hits], hits[-1]["sort"]


page1, cursor = page_cursor()
print(f"  User loads page 1  →  posts {page1}")
print(f"  Cursor handed to the client: {cursor}   ← the last row's sort key")

# Same interruption as before: 3 newer posts arrive.
helpers.bulk(es, [
    {"_index": PAGING_INDEX, "_id": 100 + j,
     "_source": {"post_id": 100 + j, "content": "coffee breaking news",
                 "created_at": (T0 + timedelta(minutes=j)).isoformat()}}
    for j in range(1, 4)])
es.indices.refresh(index=PAGING_INDEX)
print("  … the same 3 newer posts are indexed …")

page2, cursor = page_cursor(after=cursor)
print(f"  User clicks next   →  posts {page2}")

dupes = sorted(set(page1) & set(page2))
print(f"\n  ✅ Posts shown twice: {dupes or 'none'}")
assert not dupes, f"search_after still duplicated rows: {dupes}"
print("     The cursor is anchored to post 5's sort key, so the reading position")
print("     survived 3 insertions above it.")
print("""
  ⚖️  What search_after gives up:
       • No random access. There is no "jump to page 7" — only next, next, next.
         (Which is why infinite scroll and cursor pagination arrived together.)
       • Still not a snapshot. The 3 new posts are genuinely missing from this
         user's session until they refresh. If you need a frozen result set,
         open a point-in-time (`_pit`) and page inside it — at the cost of the
         server holding search contexts open, which you must then expire.
       • Going *backwards* needs a second cursor with the sort reversed.""")

In [ ]:
# ── The second problem: deep paging is expensive by construction ───────────
#
# To return rows 100,000-100,010, a coordinating node must ask EVERY shard for
# its top 100,010 and merge them — because any shard could own row 100,001.
# Cost grows with `from`, not with `size`. Elasticsearch refuses past a limit.

from elasticsearch import BadRequestError

for offset in (0, 9_990, 10_000):
    try:
        es.search(index=PAGING_INDEX, query={"match_all": {}}, from_=offset, size=10)
        print(f"  from={offset:<7} ✅ allowed")
    except BadRequestError as exc:
        reason = str(exc.body["error"]["root_cause"][0]["reason"])
        print(f"  from={offset:<7} ❌ rejected — {reason.split('.')[0]}")

print("""
  `index.max_result_window` defaults to 10,000. You can raise it. You should
  almost never want to: with S shards, `from=N` makes every shard build and
  ship a top-(N+size) list, so memory and CPU scale with N x S per query. It is
  a guard rail, not an arbitrary limit.

  The product answer is usually better than the engineering one: nobody clicks
  to page 1,000 of search results. Cap the reachable depth, and make deeper
  exploration a filter (date range, author, group) rather than an offset —
  filters shrink the result set instead of walking further into it.""")

# ── Cleanup ────────────────────────────────────────────────────────────────
es.indices.delete(index=PAGING_INDEX)
print(f"\n🧹 Deleted {PAGING_INDEX}.")

## 📊 Ranking Comparison Dashboard

Let's compare all ranking strategies side by side for the same query.

In [ ]:
# Compare ranking strategies for "python"

keyword = "python"

strategies = {
    "BM25 Only": es.search(index=INDEX_NAME, body={
        "query": {"match": {"content": keyword}}, "size": 5
    }),
    "By Recency": search_by_recency(keyword, 5),
    "By Likes": search_by_likes(keyword, 5),
    "Combined": search_combined_ranking(keyword, 5),
}

for name, result in strategies.items():
    print(f"\n{'=' * 70}")
    print(f"📊 Strategy: {name}")
    print(f"{'=' * 70}")
    for i, hit in enumerate(result["hits"]["hits"], 1):
        src = hit["_source"]
        date = src["created_at"][:10] if src["created_at"] else "unknown"
        # When sorting by a field, Elasticsearch returns _score=None, so guard against it.
        score_str = f"{hit['_score']:>8.2f}" if hit.get('_score') is not None else "    n/a "
        print(f"  #{i} | score={score_str} | {date} | {src['like_count']:>5} likes | {src['content'][:40]}...")

print("\n💡 Notice how different strategies surface different posts!")
print("   The 'right' ranking depends on what your users want.")

## 🧹 Cleanup

In [ ]:
# Clean up all Elasticsearch indices

for idx in ["posts", "posts_autocomplete", "search_suggestions"]:
    if es.indices.exists(index=idx):
        es.indices.delete(index=idx)
        print(f"🧹 Deleted index: {idx}")

print()
print("To stop all containers:")
print("  cd 06-system-designs/fb-post-search")
print("  docker compose down -v")

## 📚 Summary

### Key Takeaways

1. **BM25 measures text relevance** — it considers term frequency, document frequency, and document length. Elasticsearch uses it by default.
2. **Sorting by a field** (recency or likes) ignores relevance entirely. Simple but sometimes useful.
3. **`function_score` combines signals** — multiply BM25 with popularity boosts (log1p) and recency decay (exponential). This is the production recipe.
4. **Multi-keyword queries** have three modes: OR (broadest), AND (stricter), and phrase match (strictest). Use `bool` queries to boost exact phrases.
5. **Two-stage architecture** — use approximate scores for fast retrieval (Stage 1), then re-rank with precise data (Stage 2). This is how production systems handle stale indexes.

### How This Maps to the Facebook Post Search Design

| Concept | How It's Used |
|---------|---------------|
| Inverted Index | Maps keywords → post IDs in Redis/Elasticsearch |
| BM25 Scoring | Ranks posts by text relevance |
| Recency Sorting | Creation index sorted by timestamp |
| Like Count Sorting | Likes index using Redis sorted sets |
| Two-Stage Architecture | Approximate ranking → fetch real likes → re-sort |
| Phrase Matching | Bigram indexes or set intersection |
| Caching | CDN + Redis cache for repeated queries |

### 🎓 You've Completed the FB Post Search Lab!

You now understand the core search concepts that come up in system design interviews:
- **Notebook 1**: How inverted indexes make search fast
- **Notebook 2**: How typeahead and autocomplete work
- **Notebook 3**: How to rank and combine multiple signals

### Further Reading

- [Elasticsearch: The Definitive Guide](https://www.elastic.co/guide/en/elasticsearch/reference/current/index.html)
- [Hello Interview: FB Post Search Design](https://www.hellointerview.com/learn/system-design/problem-breakdowns/fb-post-search)
- [BM25 Algorithm Explained](https://en.wikipedia.org/wiki/Okapi_BM25)